# Topic Modeling Pipeline — LDA & BERTopic



## 1. Imports

In [ ]:
import re
import string
from dataclasses import dataclass, field
from typing import Optional
import emoji
import pandas as pd
import spacy
import gensim.corpora as corpora
from gensim.models import LdaModel
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN


## 2. Configuration


In [62]:
 
@dataclass
class DatasetConfig:
    """Describes a dataset to analyze."""
    name: str                                                        # short identifier e.g. "fr", "multi"
    path: str                                                        # path to CSV file
    text_col: str          = "text"                                  # column containing raw text
    spacy_model: str       = "fr_core_news_sm"                       # spaCy model (French only)
    embedding_model: str   = "dangvantuan/sentence-camembert-base"   # sentence-transformer model for BERTopic
 
 
@dataclass
class LdaConfig:
    num_topics: int        = 4
    passes_first: int      = 10     # quick first pass for exploration
    passes_second: int     = 30     # longer second pass after adding domain stopwords
    random_state: int      = 42
    alpha: str             = "auto"
    extra_stopwords: set[str] = field(default_factory=lambda: {
        "café", "vraiment", "super", "trop", "bon", "bien", "point", "top"
    })
 
 
@dataclass
class BertopicConfig:
    # embedding model is taken from DatasetConfig, not defined here
    nr_topics: int                = 4
    top_n_words: int              = 8
    umap_n_neighbors: int         = 5
    umap_n_components: int        = 5
    hdbscan_min_cluster_size: int = 3
    hdbscan_min_samples: int      = 2
    extra_stopwords: set[str]     = field(default_factory=lambda: {
        "café", "vraiment", "super", "trop", "bon", "bien", "point", "top"
    })
 
 
# Available datasets — add new entries here to test on other data
DATASETS = {
    "fr": DatasetConfig(
        name            = "fr",
        path            = "../data/processed/genova/genova_reviews_french.csv",
        embedding_model = "dangvantuan/sentence-camembert-base",      # French-specific model
    ),
    "multi": DatasetConfig(
        name            = "multi",
        path            = "../data/processed/genova/genova_reviews.csv",
        embedding_model = "paraphrase-multilingual-MiniLM-L12-v2",   # Multilingual model
    ),
}


## 3. Preprocessing

In [63]:
 
def text_cleaning(text: str) -> str:
    """
    Language-agnostic light cleaning pipeline.
    Keeps Latin, accented, and Arabic characters. Drops pure numbers and emojis.
    """
    text = str(text).lower()
 
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
 
    # Remove phone numbers
    text = re.sub(r'\b\+?\d[\d\s\-\(\)]{7,}\d\b', ' ', text)
 
    # Remove emojis
    text = emoji.replace_emoji(text, replace=' ')
 
    # Replace punctuation with spaces
    text = text.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
 
    # Keep only valid characters; drop purely numeric tokens
    tokens = []
    for token in text.split():
        token = re.sub(r'[^a-z\u00C0-\u017E\u0600-\u06FF0-9]', '', token)
        if token and re.search(r'[a-z\u00C0-\u017E\u0600-\u06FF]', token):
            tokens.append(token)
 
    text = ' '.join(tokens)
 
    # Normalize repeated characters (e.g. "cooool" → "cool")
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
 
    # Collapse extra whitespace
    return re.sub(r'\s+', ' ', text).strip()
 
 
def pipeline_fr(text: str, nlp) -> list[str]:
    """
    French pipeline: light cleaning → spaCy lemmatization → stopword removal.
    Returns a list of lemmatized tokens (used by LDA and BERTopic FR).
    """
    cleaned = text_cleaning(text)
    if not cleaned:
        return []
    doc = nlp(cleaned)
    tokens = []
    for token in doc:
        if not token.is_punct and not token.is_space and not token.is_stop:
            lemme = re.sub(r'[^a-z\u00C0-\u017E]', '', token.lemma_)
            if lemme and len(lemme) > 1:
                tokens.append(lemme)
    return tokens
 
 
def pipeline_fr_bertopic(text: str, nlp) -> str:
    """
    French pipeline for BERTopic: same as pipeline_fr but returns a string.
    """
    return ' '.join(pipeline_fr(text, nlp))
 
 
def pipeline_multi(text: str) -> str:
    """
    Multilingual pipeline: light cleaning only.
    No lemmatization, no stopword removal — preserves all languages.
    """
    return text_cleaning(text)


## 4. LDA

In [64]:
def _build_corpus(token_series: pd.Series):
    """Build a gensim dictionary and bag-of-words corpus from a token series."""
    id2word = corpora.Dictionary(token_series)
    corpus  = [id2word.doc2bow(tokens) for tokens in token_series]
    return id2word, corpus
 
 
def _train_lda(corpus, id2word, cfg: LdaConfig, passes: int) -> LdaModel:
    """Train and return an LDA model."""
    return LdaModel(
        corpus         = corpus,
        id2word        = id2word,
        num_topics     = cfg.num_topics,
        random_state   = cfg.random_state,
        passes         = passes,
        alpha          = cfg.alpha,
        per_word_topics= True,
    )
 
 
def _print_topics(model: LdaModel) -> None:
    print("--- LDA Topics ---")
    for idx, topic in model.print_topics(-1):
        print(f"Topic {idx}: {topic}\n")
 
def run_lda(df: pd.DataFrame, text_col: str, nlp, cfg: LdaConfig) -> LdaModel:
    """
    French  : text_cleaning → spaCy lemmatization + stopword removal
    Multi   : text_cleaning → simple tokenization (no lemmatization, no stopwords)
    """
    if nlp is not None:
        tokenize = lambda t: pipeline_fr(t, nlp)
    else:
        tokenize = lambda t: text_cleaning(t).split()

    # --- Pass 1: quick exploration ---
    print("\n" + "=" * 50)
    print("LDA — Pass 1 (exploration)")
    print("=" * 50)

    df["tokens_lda"] = df[text_col].apply(tokenize)
    id2word, corpus  = _build_corpus(df["tokens_lda"])
    model            = _train_lda(corpus, id2word, cfg, passes=cfg.passes_first)
    _print_topics(model)

    # --- Pass 2: retrain with more passes ---
    # For French: domain stopwords are added before retraining
    # For Multi : no stopwords added, just longer training
    print("\n" + "=" * 50)
    print("LDA — Pass 2 (longer training)")
    print("=" * 50)

    if nlp is not None:
        for word in cfg.extra_stopwords:
            nlp.vocab[word].is_stop = True
        tokenize = lambda t: pipeline_fr(t, nlp)

    df["tokens_lda"] = df[text_col].apply(tokenize)
    id2word, corpus  = _build_corpus(df["tokens_lda"])
    model            = _train_lda(corpus, id2word, cfg, passes=cfg.passes_second)
    _print_topics(model)

    return model

## 5. BERTopic

In [65]:
 
def run_bertopic(
    df      : pd.DataFrame,
    text_col: str,
    nlp,
    cfg     : BertopicConfig,
    ds_cfg  : DatasetConfig,
) -> BERTopic:
    """
    Train BERTopic on the dataset.
    Preprocessing and embedding model are selected automatically:
      - nlp is not None → French pipeline (lemmatized) + French embedding model
      - nlp is None     → Multilingual pipeline (light cleaning) + multilingual embedding model
    Returns the trained BERTopic model.
    """
    print("\n" + "=" * 50)
    print(f"BERTopic — dataset: {ds_cfg.name} | model: {ds_cfg.embedding_model}")
    print("=" * 50)
 
    # Select preprocessing pipeline based on language
    if nlp is not None:
        df["bertopic_text"] = df[text_col].apply(lambda t: pipeline_fr_bertopic(t, nlp))
    else:
        df["bertopic_text"] = df[text_col].apply(pipeline_multi)
 
    # Filter out empty or very short documents
    docs = [d for d in df["bertopic_text"].dropna().tolist() if len(d.strip()) > 10]
    print(f"  {len(docs)} valid documents")
 
    # Stopwords for the vectorizer: spaCy list + domain words (French only)
    base_stopwords = list(nlp.Defaults.stop_words) if nlp is not None else []
    all_stopwords  = base_stopwords + list(cfg.extra_stopwords)
 
    # Embedding model comes from the dataset config
    embedding_model = SentenceTransformer(ds_cfg.embedding_model)
 
    umap_model = UMAP(
        n_neighbors = cfg.umap_n_neighbors,
        n_components= cfg.umap_n_components,
        min_dist    = 0.0,
        metric      = "cosine",
        random_state= 42,
    )
 
    hdbscan_model = HDBSCAN(
        min_cluster_size= cfg.hdbscan_min_cluster_size,
        min_samples     = cfg.hdbscan_min_samples,
        metric          = "euclidean",
        prediction_data = True,
    )
 
    vectorizer = CountVectorizer(
        ngram_range = (1, 1),
        stop_words  = all_stopwords if all_stopwords else None,
        min_df      = 1,
    )
 
    topic_model = BERTopic(
        embedding_model = embedding_model,
        umap_model      = umap_model,
        hdbscan_model   = hdbscan_model,
        vectorizer_model= vectorizer,
        top_n_words     = cfg.top_n_words,
        nr_topics       = cfg.nr_topics,
        verbose         = True,
    )
 
    topics, _ = topic_model.fit_transform(docs)
 
    # Attach topic assignments back to the dataframe
    df_result          = df.iloc[:len(docs)].copy()
    df_result["topic"] = topics
 
    # Display results
    topic_info = topic_model.get_topic_info()[["Topic", "Count", "Name", "Representation"]]
    print("\n--- BERTopic Topics ---")
    for _, row in topic_info.iterrows():
        print(f"Topic {row['Topic']} (n={row['Count']}): {row['Representation']}")
 
    return topic_model
 

## 6. Exécution

In [66]:
ACTIVE_DATASET = "fr"
 
ds_cfg = DATASETS[ACTIVE_DATASET]
df     = pd.read_csv(ds_cfg.path)
 
# Load spaCy only for French; multilingual pipeline does not need it
nlp = spacy.load(ds_cfg.spacy_model) if ACTIVE_DATASET == "fr" else None
 
print(f"{len(df)} rows loaded — text column: '{ds_cfg.text_col}'")
print(f"Embedding model: {ds_cfg.embedding_model}")
 


49 rows loaded — text column: 'text'
Embedding model: dangvantuan/sentence-camembert-base


In [67]:
lda_cfg   = LdaConfig()
lda_model = run_lda(df.copy(), ds_cfg.text_col, nlp, lda_cfg)


LDA — Pass 1 (exploration)
--- LDA Topics ---
Topic 0: 0.026*"serveur" + 0.018*"interdiction" + 0.018*"café" + 0.018*"bien" + 0.018*"goût" + 0.018*"vraiment" + 0.018*"fumée" + 0.018*"point" + 0.018*"être" + 0.018*"espace"

Topic 1: 0.056*"café" + 0.020*"excellent" + 0.014*"bien" + 0.014*"travail" + 0.014*"parfaire" + 0.014*"top" + 0.014*"wifi" + 0.014*"staff" + 0.014*"goût" + 0.014*"journée"

Topic 2: 0.033*"service" + 0.028*"étudiant" + 0.023*"café" + 0.023*"vraiment" + 0.017*"petit" + 0.017*"bon" + 0.017*"plein" + 0.012*"non" + 0.012*"qualité" + 0.012*"table"

Topic 3: 0.040*"café" + 0.025*"étudiant" + 0.017*"service" + 0.017*"super" + 0.017*"réviser" + 0.017*"bien" + 0.017*"bon" + 0.017*"spot" + 0.017*"matin" + 0.017*"beaucoup"


LDA — Pass 2 (longer training)
--- LDA Topics ---
Topic 0: 0.026*"service" + 0.026*"étudiant" + 0.020*"goût" + 0.020*"serveur" + 0.020*"espace" + 0.020*"matin" + 0.014*"qualité" + 0.014*"réviser" + 0.014*"dommage" + 0.014*"enser"

Topic 1: 0.023*"petit" + 

In [68]:

bertopic_cfg = BertopicConfig()
bert_model   = run_bertopic(df.copy(), ds_cfg.text_col, nlp, bertopic_cfg, ds_cfg)


BERTopic — dataset: fr | model: dangvantuan/sentence-camembert-base
  48 valid documents


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-05-11 03:32:49,273 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

2026-05-11 03:32:49,572 - BERTopic - Embedding - Completed ✓
2026-05-11 03:32:49,573 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-11 03:32:49,603 - BERTopic - Dimensionality - Completed ✓
2026-05-11 03:32:49,604 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-11 03:32:49,607 - BERTopic - Cluster - Completed ✓
2026-05-11 03:32:49,608 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-05-11 03:32:49,616 - BERTopic - Representation - Completed ✓
2026-05-11 03:32:49,617 - BERTopic - Topic reduction - Reducing number of topics
2026-05-11 03:32:49,620 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-11 03:32:49,627 - BERTopic - Representation - Completed ✓
2026-05-11 03:32:49,630 - BERTopic - Topic reduction - Reduced number of topics from 7 to 4



--- BERTopic Topics ---
Topic -1 (n=2): ['tasse', 'mémorable', 'médiocre', 'lent', 'habituel', 'rien', 'service', '']
Topic 0 (n=23): ['étudiant', 'concentrer', 'serveur', 'non', 'interdiction', 'venir', 'endroit', 'wifi']
Topic 1 (n=12): ['petit', 'table', 'déjeuner', 'matin', 'service', 'journée', 'dépanner', 'commande']
Topic 2 (n=11): ['goût', 'quantité', 'prix', 'service', 'qualité', 'aimer', 'congru', 'éviter']


In [69]:
ACTIVE_DATASET = "multi"
 
ds_cfg = DATASETS[ACTIVE_DATASET]
df     = pd.read_csv(ds_cfg.path)
 
# Load spaCy only for French; multilingual pipeline does not need it
nlp = spacy.load(ds_cfg.spacy_model) if ACTIVE_DATASET == "fr" else None
 
print(f"{len(df)} rows loaded — text column: '{ds_cfg.text_col}'")
print(f"Embedding model: {ds_cfg.embedding_model}")
 


44 rows loaded — text column: 'text'
Embedding model: paraphrase-multilingual-MiniLM-L12-v2


In [70]:
lda_cfg   = LdaConfig()
lda_model = run_lda(df.copy(), ds_cfg.text_col, nlp, lda_cfg)


LDA — Pass 1 (exploration)
--- LDA Topics ---
Topic 0: 0.025*"service" + 0.017*"w" + 0.017*"مقهى" + 0.017*"the" + 0.017*"is" + 0.010*"l" + 0.010*"nadi" + 0.010*"envie" + 0.010*"b" + 0.010*"mkhyer"

Topic 1: 0.020*"مزيان" + 0.014*"service" + 0.014*"قهوة" + 0.014*"بزاف" + 0.014*"excellent" + 0.014*"وهادشي" + 0.008*"good" + 0.008*"un" + 0.008*"d" + 0.008*"envie"

Topic 2: 0.037*"the" + 0.023*"service" + 0.023*"w" + 0.019*"café" + 0.014*"le" + 0.014*"khari" + 0.014*"coffee" + 0.014*"was" + 0.010*"is" + 0.010*"على"

Topic 3: 0.028*"l" + 0.017*"قهوة" + 0.017*"w" + 0.012*"القهوة" + 0.012*"بسيطة" + 0.012*"على" + 0.012*"من" + 0.012*"bien" + 0.012*"et" + 0.007*"qahwa"


LDA — Pass 2 (longer training)
--- LDA Topics ---
Topic 0: 0.025*"service" + 0.017*"w" + 0.017*"مقهى" + 0.017*"the" + 0.017*"is" + 0.010*"l" + 0.010*"nadi" + 0.010*"envie" + 0.010*"3amra" + 0.010*"kay3tik"

Topic 1: 0.020*"مزيان" + 0.014*"service" + 0.014*"قهوة" + 0.014*"بزاف" + 0.014*"excellent" + 0.014*"وهادشي" + 0.008*"good" 

In [71]:
bertopic_cfg = BertopicConfig()
bert_model   = run_bertopic(df.copy(), ds_cfg.text_col, nlp, bertopic_cfg, ds_cfg)


BERTopic — dataset: multi | model: paraphrase-multilingual-MiniLM-L12-v2
  41 valid documents


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-05-11 03:32:55,116 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

2026-05-11 03:32:55,346 - BERTopic - Embedding - Completed ✓
2026-05-11 03:32:55,347 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-11 03:32:55,371 - BERTopic - Dimensionality - Completed ✓
2026-05-11 03:32:55,372 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-11 03:32:55,376 - BERTopic - Cluster - Completed ✓
2026-05-11 03:32:55,376 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-05-11 03:32:55,383 - BERTopic - Representation - Completed ✓
2026-05-11 03:32:55,383 - BERTopic - Topic reduction - Reducing number of topics
2026-05-11 03:32:55,383 - BERTopic - Topic reduction - Number of topics (4) is equal or higher than the clustered topics(4).
2026-05-11 03:32:55,384 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-11 03:32:55,430 - BERTopic - Representation - Completed ✓



--- BERTopic Topics ---
Topic 0 (n=6): ['ممنوع', 'التدخين', 'المكان', 'أن', 'and', 'راحتهم', 'داخل', 'الناس']
Topic 1 (n=22): ['the', 'service', 'excellent', 'le', 'القهوة', 'coffee', 'was', 'قهوة']
Topic 2 (n=3): ['na9ssa', 'l9ahwa', 'khedma', 'basita', 'tebla', '7etto', '9ahwa', '']
Topic 3 (n=10): ['qahwa', 'service', 'nadi', 'ftour', 'li', 'walakin', 'wa3ra', 'hadchi']
